In [6]:
# This generates langgraph code from graph_spec
from graph_gen.gen_graph import gen_graph
%run graph_gen/common_imports.py

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("tell me a joke about {topic} and food")
model = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

In [14]:
from langchain_core.runnables import RunnablePassthrough

chain1 = prompt | model | parser
chain2 = ({"topic": RunnablePassthrough()} | prompt | model | parser)
result1 = chain1.invoke("dogs")
result2 = chain2.invoke("cats")
result3 = chain1.invoke({"topic": "gorillas"})
print(result1)
print("-"*40)
print(result2)
print("-"*40)
print(result3)

Why did the dog sit in the shade while eating his dinner?

Because he didn’t want to become a hot dog!
----------------------------------------
Why did the cat sit on the computer?

Because it wanted to keep an eye on the mouse and the recipe for its favorite tuna tart!
----------------------------------------
Why did the gorilla bring a ladder to the restaurant?

Because he heard the food was on the "top shelf"!


In [25]:
from langchain_core.runnables import RunnablePassthrough
prompt = ChatPromptTemplate.from_template("tell me a joke about {topic} and {topic2}")
model = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

chain1 = prompt | model | parser
result1 = chain1.invoke({"topic":"dogs", "topic2":"salsa"})
print(result1)

prompt = prompt.partial(topic2="food")
print("-----")
print(prompt)
chain2 = ({"topic": RunnablePassthrough()} | prompt | model | parser)
result2 = chain2.invoke("cats")
print("-----")
print(result2)
result3 = chain2.invoke({"topic": "lizards"})
print("-----")
print(result3)

Why did the dog sit in the salsa?

Because he wanted to be a little "paws" and a little "spice"! 🌶️🐾
-----
input_variables=['topic'] partial_variables={'topic2': 'food'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic', 'topic2'], template='tell me a joke about {topic} and {topic2}'))]
-----
Why did the cat sit on the bowl of food?

Because it wanted to be a "purr-fect" meal!
-----
Why did the lizard refuse to eat at the buffet?

Because he heard it was all "scale" and no "tail"!


In [36]:
from typing import Annotated, Dict, List, Optional, TypedDict, Tuple, Sequence, Union, Literal
import re
from langchain_core.pydantic_v1 import BaseModel, Field

class PersonWish(BaseModel):
    name: str = Field(description="The name of the person desiring something")
    desire: str = Field(description="What it is they desire")

class Recipe(BaseModel):
    title: str = Field(description="What we call the end result of this recipe")
    ingredients: List[str] = Field(description="the name of each ingredient and measurement needed in the recipe")

model = ChatOpenAI(model="gpt-4o-mini")
structured_model = model.with_structured_output(PersonWish)
result = structured_model.invoke("The three of them went to a car show.  But Didi was thinking about eating sushi.")
print(result)

name='Didi' desire='eating sushi'


In [37]:
tools=[PersonWish,Recipe]
model = ChatOpenAI(model="gpt-4o-mini")
model_with_tools = model.bind_tools(tools)
result = model_with_tools.invoke("The three of them went to a car show.  But Didi was thinking about eating sushi, and mark was thinking about making sushi with three small pieces of tuna, a cup of rice, all wrapped with seaweed")
print(result)

content='' additional_kwargs={'tool_calls': [{'id': 'call_HTq8LtDVRfbZkj0ce0kRoBGa', 'function': {'arguments': '{"name": "Didi", "desire": "to eat sushi"}', 'name': 'PersonWish'}, 'type': 'function'}, {'id': 'call_95g0axefbwFV2F1Yk5p2pLNl', 'function': {'arguments': '{"title": "Sushi with Tuna", "ingredients": ["3 small pieces of tuna", "1 cup of rice", "seaweed (nori)"]}', 'name': 'Recipe'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 142, 'total_tokens': 218}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_661538dc1f', 'finish_reason': 'tool_calls', 'logprobs': None} id='run-0d890bfb-aec4-44b8-88d5-3770b6a25acd-0' tool_calls=[{'name': 'PersonWish', 'args': {'name': 'Didi', 'desire': 'to eat sushi'}, 'id': 'call_HTq8LtDVRfbZkj0ce0kRoBGa', 'type': 'tool_call'}, {'name': 'Recipe', 'args': {'title': 'Sushi with Tuna', 'ingredients': ['3 small pieces of tuna', '1 cup of rice', 'seaweed (nori)']}, 'id': 'call_95g0axe

In [38]:
print(result.tool_calls)

[{'name': 'PersonWish', 'args': {'name': 'Didi', 'desire': 'to eat sushi'}, 'id': 'call_HTq8LtDVRfbZkj0ce0kRoBGa', 'type': 'tool_call'}, {'name': 'Recipe', 'args': {'title': 'Sushi with Tuna', 'ingredients': ['3 small pieces of tuna', '1 cup of rice', 'seaweed (nori)']}, 'id': 'call_95g0axefbwFV2F1Yk5p2pLNl', 'type': 'tool_call'}]


In [13]:
class State(TypedDict):
    code: str
    executed: bool
    error: str
    what_next: str

def starter(state):
    code = input("Enter code to try: ")
    return { "code": code, "executed": False }

def human_choice(state):
    action = input("'x' to execute, 'q' to quit")
    return { "what_next": action }

def is_done(state):
    return state["what_next"] == 'q'

def executor(state):
    error = None
    try:
        result = exec(state["code"])
    except Exception as e:
        error = e
    return { "executed": True, "error": error }
        
graph_spec = """
starter(State) => human_choice

human_choice => executor

executor 
   is_done => END
   => starter"""

coder_code = gen_graph("coder", graph_spec)
print(coder_code)
exec(coder_code)

coder = StateGraph(State)
coder.add_node('starter', starter)
coder.add_node('human_choice', human_choice)
coder.add_node('executor', executor)

coder.set_entry_point('starter')

coder.add_edge('starter', 'human_choice')
coder.add_edge('human_choice', 'executor')
def after_executor(state: State):
    if is_done(state):
        return 'END'
    return 'starter'

executor_dict = {'END': END, 'starter': 'starter'}
coder.add_conditional_edges('executor', after_executor, executor_dict)


coder = coder.compile()


In [ ]:
for s in coder.stream({"code": None}, stream_mode="updates"):
    print(s)

Enter code to try:  print('hi')


{'starter': {'code': "print('hi')", 'executed': False}}


'x' to execute, 'q' to quit x


{'human_choice': {'what_next': 'x'}}
hi
{'executor': {'executed': True, 'error': None}}


Enter code to try:  print hi


...

{'human_choice': {'what_next': 'x'}}
{'executor': {'executed': True, 'error': SyntaxError("Missing parentheses in call to 'print'. Did you mean print(...)?", ('<string>', 1, 1, 'print hi\n', 1, 9))}}


Enter code to try:  print("This is a test")


{'starter': {'code': 'print("This is a test")', 'executed': False}}


'x' to execute, 'q' to quit x


{'human_choice': {'what_next': 'x'}}
This is a test
{'executor': {'executed': True, 'error': None}}


In [53]:
class ComplianceRule(BaseModel):
    rule: str = Field(description="a memorable name for the rule")
    checks: list[str] = Field(description="list of detailed conditions that must be met for the rule to pass")
    passed: bool = Field(description="did this rule pass")
    reasoning: str = Field(description="brief justification for the pass or fail result")

class EmployeeInTrouble(BaseModel):
    name: str = Field(description="name of the employee")
    violation: str = Field(description="the safety violation they need to be aware of")

class EmployeesInTrouble(BaseModel):
    employees: List[EmployeeInTrouble] = Field(description="the employees that have violated a rule")

rule = """Our cleanliness rule:  
All employees must wash their hands before leaving the restroom, 
as well as after shaking hands with customers.  In addition, any utensil handled by an employee must be washed before used by another employee.

In today's scenario, Bob takes a dump, then exits the bathroom, then immediately picks up a knife, cuts some tomatoes, then
hands the knife off to Silvia."""

model = ChatOpenAI(model="gpt-4o-mini")
structured_model = model.with_structured_output(ComplianceRule)
result = structured_model.invoke(rule)
print(result)

rule='Cleanliness Rule Compliance' checks=['Bob washed his hands after using the restroom', 'Bob washed the knife before handing it to Silvia'] passed=False reasoning='Bob did not wash his hands after using the restroom before handling the knife, and he also did not wash the knife before handing it off to Silvia.'


In [54]:
model = ChatOpenAI(model="gpt-4o-mini")
tool_model = model.bind_tools([ComplianceRule,EmployeeInTrouble])
result = tool_model.invoke(rule)
result.tool_calls

[{'name': 'ComplianceRule',
  'args': {'rule': 'Cleanliness Rule Compliance',
   'checks': ['Bob washed his hands after using the restroom',
    'Bob washed hands after shaking hands with customers',
    'Knife was washed before being handed to Silvia'],
   'passed': False,
   'reasoning': 'Bob did not wash his hands after leaving the restroom and before handling the knife.'},
  'id': 'call_bA0FTgCafAVKZZtvEdeoOc4q',
  'type': 'tool_call'},
 {'name': 'EmployeeInTrouble',
  'args': {'name': 'Bob',
   'violation': 'Did not wash hands after using the restroom before handling kitchen utensils.'},
  'id': 'call_ZwUzuX912bD2LqEALnY7qGCz',
  'type': 'tool_call'}]

In [55]:
model = ChatOpenAI(model="gpt-4o-mini")
tool_model = model.bind_tools([ComplianceRule,EmployeesInTrouble])
result = tool_model.invoke(rule)
result.tool_calls

[{'name': 'ComplianceRule',
  'args': {'rule': 'Cleanliness Rule',
   'checks': ['Bob must wash hands after using the restroom',
    'Bob must wash hands after cutting tomatoes and before handling utensils',
    'The knife must be washed before being handed to Silvia'],
   'passed': False,
   'reasoning': 'Bob did not wash his hands after using the restroom and before handling the knife, and the knife was not washed before being handed to Silvia.'},
  'id': 'call_hiscLlL7bQ7bTvPAJvgJAZVS',
  'type': 'tool_call'},
 {'name': 'EmployeesInTrouble',
  'args': {'employees': [{'name': 'Bob',
     'violation': 'Did not wash hands after using the restroom and before handling the knife.'},
    {'name': 'Silvia',
     'violation': 'Received a knife that was not washed after being used by Bob.'}]},
  'id': 'call_BKKn60tnSdUqN5A8a82Mjprb',
  'type': 'tool_call'},
 {'name': 'ComplianceRule',
  'args': {'rule': 'Utensil Handling Rule',
   'checks': ['The knife must be washed before being used by anot

In [50]:
rule = """ 
All employees must wash their hands before leaving the restroom, 
as well as after shaking hands with customers.  In addition, any utensil handled by an employee must be washed before used by another employee."""

scenario = """
In today's scenario, Bob takes a dump, then exits the bathroom, then immediately picks up a knife, cuts some tomatoes, then
hands the knife off to Silvia.  Timmy shakes a customer's hand, then washes his hands immediately after saying 'Ewww'. """

prompt = """We are checking for compliance to rules, and tracking the violations of those rules.

RULES:
   {rules}

SCENARIO:
   {scenario}
"""

prompt = ChatPromptTemplate.from_template(scenario)
prompt = prompt.partial(rules=rule)
model = ChatOpenAI(model="gpt-4o-mini").bind_tools([ComplianceRule,EmployeeInTrouble])
chain = prompt | model
result = chain.invoke(scenario)
result

NameError: name 'EmployeeInTrouble' is not defined